<div style="background: linear-gradient(135deg, #0d1b2a 0%, #1b2f4b 50%, #1a5276 100%); padding: 50px 40px; border-radius: 16px; text-align: center; margin-bottom: 30px;">
  <h1 style="color: #e0f2fe; font-size: 2.8em; font-weight: 800; letter-spacing: 2px; margin: 0;">📊 SEABORN MASTERCLASS</h1>
  <h2 style="color: #7ec8e3; font-size: 1.6em; font-weight: 400; margin-top: 10px;">Notebook 4 — Multivariate Analysis</h2>
  <p style="color: #ccc; font-size: 1em; margin-top: 18px;">heatmap · pairplot · jointplot · clustermap · missing value analysis</p>
  <hr style="border: 1px solid #7ec8e3; margin: 20px auto; width: 60%;">
  <p style="color: #aaa; font-size: 0.9em;">📦 Library: <strong style='color:#e0f2fe'>Seaborn</strong> &nbsp;|&nbsp; ⏱ Est. Time: 110 mins &nbsp;|&nbsp; 🎯 Fresher Interview Focus</p>
</div>

---

## 📋 Table of Contents

| # | Section | Plots / Skills Covered |
|---|---------|----------------------|
| 1 | [Learning Objectives](#objectives) | — |
| 2 | [Quick Reference Card](#qrc) | All 4 plots at a glance |
| 3 | [Setup & Data Loading](#setup) | — |
| 4 | [heatmap — Correlation Matrices](#heatmap) | `sns.heatmap` |
| 5 | [Missing Value Heatmap](#missing) | `df.isnull()` + heatmap |
| 6 | [pairplot — All-vs-All Relationships](#pairplot) | `sns.pairplot` |
| 7 | [jointplot — Bivariate Deep Dive](#jointplot) | `sns.jointplot` |
| 8 | [clustermap — Hierarchical Grouping](#clustermap) | `sns.clustermap` |
| 9 | [Feature Correlation Ranking](#feat-corr) | Pandas + Seaborn |
| 10 | [Business Dashboard](#dashboard) | Multi-panel report |
| 11 | [Best Practices](#best-practices) | — |
| 12 | [Common Mistakes](#common-mistakes) | — |
| 13 | [Interview Notes](#interview-notes) | — |
| 14 | [Summary & Key Takeaways](#summary) | — |
| 15 | [Practice Questions](#practice) | 15 exercises |

---

<a id='objectives'></a>
## 🎯 Learning Objectives

By the end of this notebook you will be able to:

- ✅ Build a **correlation heatmap** with masked upper triangle — the #1 most-asked Seaborn technique in interviews
- ✅ Detect and visualize **missing values** using a heatmap of `df.isnull()`
- ✅ Create a `pairplot` with `hue=` and compare `diag_kind='kde'` vs `'hist'`
- ✅ Use `jointplot` to do a deep bivariate analysis with scatter, KDE, regression, and hexbin views
- ✅ Interpret a `clustermap` dendrogram to find natural feature groupings
- ✅ Rank features by correlation with a target variable — a core ML feature selection skill
- ✅ Understand **when NOT** to use each multivariate plot
- ✅ Answer every interview question about correlation, heatmaps, and pairplots confidently

**⏱ Estimated Time:** ~110 minutes | **📋 Prerequisites:** Notebooks 1–3 completed

---

<a id='qrc'></a>
## ⚡ Quick Reference Card

> 🎯 **Fresher Interview Focus** — These 4 plots are the complete toolkit for multivariate EDA.

| Plot | Minimum Syntax | 🔑 Key Interview Parameter |
|------|---------------|---------------------------|
| `heatmap` | `sns.heatmap(df.corr(), annot=True, fmt='.2f')` | `mask=np.triu(...)` → hide redundant upper triangle |
| `pairplot` | `sns.pairplot(df, hue='cat_col')` | `diag_kind='kde'` → smoother + shows overlap between groups |
| `jointplot` | `sns.jointplot(data=df, x='x', y='y', kind='reg')` | `kind=` → `'scatter'`, `'kde'`, `'hex'`, `'reg'`, `'resid'` |
| `clustermap` | `sns.clustermap(df.corr(), annot=True)` | `standard_scale=1` → normalize before clustering |

### 🔑 Critical Heatmap Mask Pattern (memorize this)
```python
import numpy as np
corr = df.select_dtypes('number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))   # True = hide
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True)
```

### 🔑 Feature Correlation with Target (memorize this)
```python
corr_target = df.corr()['target_col'].drop('target_col').abs()
                .sort_values(ascending=False)
```

### 🚦 Difficulty Legend
| Marker | Level | Meaning |
|--------|-------|---------|
| 🟢 Basic | Must-know | Core syntax — asked in every interview |
| 🟡 Intermediate | Good-to-know | Parameters and combinations |
| 🔴 Advanced | Stand-out | Production patterns that impress |

---

<a id='setup'></a>
## 1. Setup & Data Loading

In [ ]:
# 🟢 Basic | Standard setup — same across all notebooks
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

sns.set_theme(
    style='whitegrid',
    context='notebook',
    palette='deep',
    font_scale=1.1,
    rc={
        'figure.facecolor': 'white',
        'axes.facecolor':   'white',
        'grid.alpha':        0.4,
        'figure.dpi':        110,
    }
)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
pd.set_option('display.max_columns', None)
os.makedirs('outputs', exist_ok=True)
print("✅ Setup complete.")

In [ ]:
# 🟢 Basic | Load datasets ONCE — reuse throughout
# Business domains:
#   tips      → Restaurant analytics        (heatmap, pairplot)
#   penguins  → Biological classification   (pairplot, clustermap)
#   mpg       → Automotive engineering      (correlation ranking)
#   titanic   → Survival analysis           (missing values)
#   diamonds  → Luxury retail               (pairplot, jointplot)

tips     = sns.load_dataset('tips')
penguins = sns.load_dataset('penguins').dropna()
mpg      = sns.load_dataset('mpg').dropna()
titanic  = sns.load_dataset('titanic')   # Keep NaN — for missing value demo
diamonds = sns.load_dataset('diamonds')

# Engineer useful columns
tips['tip_pct']     = (tips['tip'] / tips['total_bill'] * 100).round(2)
tips['table_size']  = tips['size']
mpg['efficiency']   = (mpg['mpg'] / mpg['weight'] * 1000).round(3)

print("✅ Datasets loaded:")
print(f"   tips     → {tips.shape[0]:>3} rows × {tips.shape[1]} cols")
print(f"   penguins → {penguins.shape[0]:>3} rows × {penguins.shape[1]} cols")
print(f"   mpg      → {mpg.shape[0]:>3} rows × {mpg.shape[1]} cols")
print(f"   titanic  → {titanic.shape[0]:>3} rows × {titanic.shape[1]} cols")
print(f"   diamonds → {diamonds.shape[0]:>5} rows × {diamonds.shape[1]} cols")
print()
print("📊 Titanic missing values (for demo):")
missing_counts = titanic.isnull().sum()
print(missing_counts[missing_counts > 0].to_string())

### 📋 Output Expected
```
✅ Datasets loaded:
   tips     →  244 rows × 9 cols
   penguins →  333 rows × 8 cols
   mpg      →  392 rows × 9 cols
   titanic  →  891 rows × 15 cols
   diamonds → 53940 rows × 10 cols

📊 Titanic missing values (for demo):
age          177
embarked       2
embark_town    2
deck         688
```

---

<a id='heatmap'></a>
## 2. `heatmap` — Correlation Matrices

### 🧠 Intuition

A **correlation heatmap** answers: *"Which variables are related to each other?"*

It shows the **Pearson correlation coefficient** between every pair of numeric variables, encoded as color:
- 🔴 Red → strong positive correlation (+1)
- ⬜ White → no correlation (0)
- 🔵 Blue → strong negative correlation (−1)

### 📌 Syntax & Parameters
```python
sns.heatmap(
    data       = corr_matrix,     # Must be a square correlation matrix
    annot      = True,            # Show numbers inside cells
    fmt        = '.2f',           # Format of annotations ('2 decimal places')
    cmap       = 'coolwarm',      # Diverging colormap: blue→white→red
    center     = 0,               # Center the colormap at 0
    vmin       = -1,              # Minimum color scale value
    vmax       = 1,               # Maximum color scale value
    square     = True,            # Force square cells
    linewidths = 0.5,             # Thin grid lines between cells
    mask       = mask_array,      # Boolean array: True = hide that cell
    cbar_kws   = {'shrink': 0.8}, # Colorbar customization
    ax         = ax,              # Axes-level — accepts ax=
)
```

### 📊 When NOT to Use heatmap
| Situation | Problem | Alternative |
|-----------|---------|-------------|
| More than 20 variables | Matrix becomes unreadable | `clustermap` + select top features |
| Categorical features included | Pearson r is meaningless for categories | Use only `select_dtypes('number')` |
| Time series data | Correlation ignores temporal ordering | Line plots + lag analysis |
| Checking clusters | Static matrix hides natural groupings | `clustermap` |

In [ ]:
# 🟢 Basic | heatmap — correlation matrix starter
# Business Problem: Which restaurant metrics are related?
# Before building any prediction model, always check feature correlations.

tips_numeric = tips.select_dtypes(include='number')   # Only numeric columns
corr = tips_numeric.corr()

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.8,
    linecolor='white',
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
    ax=ax
)

ax.set_title(
    '🍽️ Restaurant Analytics — Feature Correlation Matrix\n'
    '(Color: Red=positive | Blue=negative | White=no correlation)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("📊 INSIGHTS:")
print(f"   total_bill ↔ tip:       r = {corr.loc['total_bill','tip']:.3f}  — strong positive")
print(f"   total_bill ↔ tip_pct:   r = {corr.loc['total_bill','tip_pct']:.3f} — weak negative")
print(f"   size ↔ total_bill:       r = {corr.loc['size','total_bill']:.3f}  — moderate positive")
print()
print("🚨 INTERVIEW TRAP: The upper triangle is a MIRROR of the lower triangle.")
print("   That's why we always mask it — next cell shows the professional standard.")

### 📋 Output Preview
```
📊 INSIGHTS:
   total_bill ↔ tip:       r = 0.676  — strong positive
   total_bill ↔ tip_pct:   r = -0.139 — weak negative
   size ↔ total_bill:       r = 0.598  — moderate positive

🚨 INTERVIEW TRAP: The upper triangle is a MIRROR of the lower triangle.
```
> 🎯 **Interview Q:** *"What is wrong with showing the full correlation heatmap?"*
> → The **upper triangle is identical** to the lower triangle — it's redundant and doubles the visual noise. **Always mask the upper triangle** in production.

In [ ]:
# 🟢 Basic | Masked upper triangle — the PROFESSIONAL standard (memorize this!)
# This is the MOST asked Seaborn pattern in fresher interviews.
# Rule: Always mask. Always use coolwarm. Always set center=0. Always square=True.

penguins_numeric = penguins.select_dtypes(include='number')
corr_peng = penguins_numeric.corr()

# ── The mask: True = hide, False = show
# np.triu creates a matrix with True in upper triangle + diagonal
mask = np.triu(np.ones_like(corr_peng, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Full matrix (bad practice) ──
sns.heatmap(
    corr_peng,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.8,
    ax=axes[0]
)
axes[0].set_title(
    '❌ Full Matrix (Bad Practice)\nUpper triangle is redundant mirror',
    fontweight='bold', color='#c62828', fontsize=11
)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

# ── Right: Masked lower triangle (professional standard) ──
sns.heatmap(
    corr_peng,
    mask=mask,              # ← THE KEY PARAMETER
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.8,
    cbar_kws={'label': 'Pearson r', 'shrink': 0.75},
    ax=axes[1]
)
axes[1].set_title(
    '✅ Masked Lower Triangle (Professional Standard)\nClean, no redundancy',
    fontweight='bold', color='#2e7d32', fontsize=11
)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

fig.suptitle('🐧 Penguin Feature Correlations — Full vs Masked Heatmap',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("📊 KEY CORRELATIONS (Penguin Dataset):")
print(f"   flipper_length ↔ body_mass:  r = {corr_peng.loc['flipper_length_mm','body_mass_g']:.3f}  ✅ strong — big penguins have longer flippers")
print(f"   bill_length ↔ bill_depth:    r = {corr_peng.loc['bill_length_mm','bill_depth_mm']:.3f} — weak negative (species mix effect)")
print(f"   bill_length ↔ flipper:       r = {corr_peng.loc['bill_length_mm','flipper_length_mm']:.3f}  — moderate positive")

### 📋 Output Preview — Masked Heatmap
```
📊 KEY CORRELATIONS:
   flipper_length ↔ body_mass:  r = 0.873  ✅ strong positive
   bill_length ↔ bill_depth:    r = -0.229 — weak negative
   bill_length ↔ flipper:       r = 0.653  — moderate positive
```
> 🏆 **The pattern to memorize:**
> ```python
> mask = np.triu(np.ones_like(corr, dtype=bool))
> sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
>             cmap='coolwarm', center=0, square=True)
> ```
> This is 5 lines. Interviewers will ask you to write it live.

In [ ]:
# 🟡 Intermediate | Styled correlation heatmap — production quality
# Business Problem: Which automotive features are most interdependent?
# Used before ML to detect multicollinearity — a key concept for freshers to know.

mpg_numeric = mpg.select_dtypes(include='number')
corr_mpg    = mpg_numeric.corr()
mask_mpg    = np.triu(np.ones_like(corr_mpg, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))

hm = sns.heatmap(
    corr_mpg,
    mask=mask_mpg,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',           # Red=negative, Yellow=neutral, Green=positive
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=1.0,
    linecolor='white',
    annot_kws={'size': 10, 'weight': 'bold'},
    cbar_kws={'label': 'Pearson Correlation (r)', 'shrink': 0.75},
    ax=ax
)

ax.set_title(
    '🚗 Automotive Analytics — Feature Correlation Matrix\n'
    '(Green = positive | Red = negative | Masked upper triangle)',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

print("📊 KEY INSIGHTS for ML Modeling:")
print(f"   mpg ↔ weight:        r = {corr_mpg.loc['mpg','weight']:.3f}  🔴 strong negative — heavier = less efficient")
print(f"   mpg ↔ cylinders:     r = {corr_mpg.loc['mpg','cylinders']:.3f}  🔴 strong negative — more cylinders = worse mpg")
print(f"   mpg ↔ horsepower:    r = {corr_mpg.loc['mpg','horsepower']:.3f}  🔴 strong negative — more power = worse mpg")
print(f"   weight ↔ horsepower: r = {corr_mpg.loc['weight','horsepower']:.3f}  ⚠️  MULTICOLLINEARITY — these two are nearly same feature!")
print()
print("⚠️  MULTICOLLINEARITY WARNING: weight, cylinders, displacement, horsepower are ALL")
print("   strongly correlated. Including all of them in a Linear Regression would be problematic.")
print("   → This is called multicollinearity — and this heatmap is how you DETECT it.")

### 📋 Output Preview — Multicollinearity Detection
```
📊 KEY INSIGHTS for ML Modeling:
   mpg ↔ weight:        r = -0.832  🔴 strong negative
   mpg ↔ cylinders:     r = -0.776  🔴 strong negative
   mpg ↔ horsepower:    r = -0.778  🔴 strong negative
   weight ↔ horsepower: r = 0.864   ⚠️  MULTICOLLINEARITY
```
> 🎯 **Interview Q:** *"What is multicollinearity and how do you detect it?"*
> → Multicollinearity is when two or more **predictor variables are highly correlated with each other** (not just with the target). It makes Linear Regression coefficients unstable. You detect it using a **correlation heatmap** — any r > 0.8 between predictors is a red flag.

---

<a id='missing'></a>
## 3. Missing Value Heatmap — Critical EDA Step

### 🧠 Why This Matters

Every real-world dataset has missing values. Before doing ANY analysis or modeling, you must:
1. **Know which columns** have missing values
2. **Know how much** is missing (percentage)
3. **Know if the pattern is random** or systematic (MCAR/MAR/MNAR)

A missing value heatmap answers question 3 visually — it shows if missingness clusters in certain rows or columns.

> 💡 **Interview Insight:** Missing data that's NOT random (MNAR - Missing Not At Random) can bias your model. For example, wealthy Titanic passengers may have had their cabin deck recorded more often — the missingness is related to wealth.

In [ ]:
# 🟡 Intermediate | Missing value heatmap — standard EDA step
# Business Problem: Titanic dataset — understand missingness patterns
# before imputation. Is the data missing randomly or systematically?

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: Boolean heatmap — white=present, black=missing ──
sns.heatmap(
    titanic.isnull(),          # ← key: df.isnull() creates boolean mask
    yticklabels=False,         # Row index not useful here (891 rows)
    cbar=False,                # No colorbar needed — binary (True/False)
    cmap='gray_r',             # White = present, Dark = missing
    ax=axes[0]
)
axes[0].set_title(
    '🚢 Titanic — Missing Value Map\n'
    '(Dark = Missing | White = Present)',
    fontweight='bold', fontsize=11
)
axes[0].set_xlabel('Columns', fontsize=11)

# ── Right: Percentage bar of missingness per column ──
miss_pct = (titanic.isnull().mean() * 100).sort_values(ascending=True)
miss_pct_nonzero = miss_pct[miss_pct > 0]

colors_bar = ['#E53935' if v > 30 else '#FB8C00' if v > 5 else '#FDD835'
              for v in miss_pct_nonzero.values]

bars = axes[1].barh(
    miss_pct_nonzero.index,
    miss_pct_nonzero.values,
    color=colors_bar,
    edgecolor='white',
    linewidth=0.8
)
axes[1].bar_label(
    bars,
    labels=miss_pct_nonzero.round(1).astype(str).add('%').tolist(),
    fontsize=11, fontweight='bold', padding=4
)
axes[1].set_title(
    'Missingness % per Column\n(Red > 30% | Orange > 5% | Yellow ≤ 5%)',
    fontweight='bold', fontsize=11
)
axes[1].set_xlabel('% of Missing Values', fontsize=11)
axes[1].axvline(x=30, color='#E53935', linestyle='--', alpha=0.6, label='30% threshold')
axes[1].axvline(x=5,  color='#FB8C00', linestyle='--', alpha=0.6, label='5% threshold')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, miss_pct_nonzero.max() * 1.15)

fig.suptitle('🔍 Exploratory Data Analysis — Missing Value Audit',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("📊 MISSING VALUE REPORT:")
print(miss_pct_nonzero.to_frame('miss_%').round(1).to_string())
print()
print("⚠️  'deck' is missing 77.2% of values — DROP this column (too sparse to impute).")
print("   'age' is missing 19.9% — IMPUTE with median grouped by class (not overall median).")
print("   'embarked' is missing 0.2% — IMPUTE with mode (only 2 rows).")

### 📋 Output Preview — Missing Data Audit
```
📊 MISSING VALUE REPORT:
         miss_%
embarked    0.2   → Impute with mode
age        19.9   → Impute with median by class
deck       77.2   → Drop column (unusable)
```
> 🎯 **Interview Q:** *"How do you handle missing values in a dataset?"*
> 1. First **visualize** with `df.isnull()` heatmap to understand the pattern
> 2. **< 5% missing** → Impute with mean/median/mode
> 3. **5–30% missing** → Impute carefully (median by group, or KNN imputation)
> 4. **> 30% missing** → Consider dropping the column
> 5. **Systematic pattern** → Investigate WHY it's missing — may be informative

---

<a id='pairplot'></a>
## 4. `pairplot` — All-vs-All Relationships

### 🧠 Intuition

A `pairplot` creates a **grid of plots** — every numeric variable plotted against every other numeric variable. It's the fastest way to get a complete overview of your data.

- **Diagonal cells**: Distribution of each variable (histogram or KDE)
- **Off-diagonal cells**: Scatter plot of one variable vs another

### 📌 Syntax & Parameters
```python
sns.pairplot(
    data       = df,
    hue        = 'cat_col',      # Color by category — reveals group structure
    vars       = ['col1','col2'], # Select specific columns (don't use all if > 8)
    diag_kind  = 'kde',          # 'auto' | 'hist' | 'kde' — diagonal plot type
    plot_kws   = {'alpha': 0.5}, # kwargs for off-diagonal scatter plots
    diag_kws   = {'fill': True}, # kwargs for diagonal plots
    corner     = True,           # Show only lower triangle (like masked heatmap)
    palette    = 'colorblind',
    height     = 2.5,            # Height of each cell in inches
    aspect     = 1.0,
)
```

### 📊 When NOT to Use pairplot
| Situation | Problem | Alternative |
|-----------|---------|-------------|
| More than 8 variables | Grid becomes tiny and unreadable | Select top correlated features first |
| Mixed numeric + categorical | Categorical columns break it | Use `select_dtypes` or `vars=` |
| n > 50,000 rows | Overplotted scatter cells | Sample first or use `kind='kde'` |

In [ ]:
# 🟢 Basic | pairplot — overview of all numeric relationships
# Business Problem: Quickly understand ALL relationships in penguin data.
# This is the first plot you run in a new dataset — takes 1 line.

g = sns.pairplot(
    data=penguins.select_dtypes(include='number'),   # Numeric only
    diag_kind='hist',           # Histogram on diagonal
    plot_kws={'alpha': 0.5, 's': 30, 'color': '#1565C0'},
    diag_kws={'color': '#1565C0', 'alpha': 0.7, 'bins': 20},
    height=2.4,
    aspect=1.0
)
g.figure.suptitle(
    '🐧 Penguin Dataset — Pairplot (All Numeric Features)\n'
    'Diagonal = Histogram | Off-diagonal = Scatter',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

print("📊 HOW TO READ A PAIRPLOT:")
print("   - Each cell shows the scatter of one feature pair (row × column label)")
print("   - Diagonal shows the distribution of each individual feature")
print("   - Strong linear scatter patterns → high correlation")
print("   - Clear clusters in scatter → likely multiple groups (species) present")

### 📋 Output Preview — Basic Pairplot
```
4×4 grid (4 numeric features):
  Diagonal:      histogram of each feature
  Off-diagonal:  scatter plots

Visual patterns to look for:
  ✅ Oval/ellipse shape → linear correlation
  ✅ Circular cloud → no correlation
  ✅ Clusters → multiple groups in data
  ✅ Curved patterns → non-linear relationship
```

In [ ]:
# 🟡 Intermediate | pairplot + hue — the most informative version
# Business Problem: Do penguin species form separate clusters?
# This is core ML: can we visually separate classes before modeling?

g = sns.pairplot(
    data=penguins,
    hue='species',                    # Color by species — reveals clusters
    diag_kind='kde',                  # KDE on diagonal — smoother, shows overlap
    plot_kws={'alpha': 0.60, 's': 35},
    diag_kws={'fill': True, 'alpha': 0.45},
    palette='colorblind',
    vars=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'],
    height=2.5,
    aspect=1.0
)

g.figure.suptitle(
    '🐧 Penguin Species Separation — Pairplot with hue\n'
    '(Separable clusters → good ML classification candidates)',
    fontsize=13, fontweight='bold', y=1.02
)

# Add a note about corner= parameter
g.figure.text(0.01, 0.01,
    "💡 Tip: Use corner=True to show only lower triangle (like masked heatmap)",
    fontsize=9, color='gray', style='italic')

plt.tight_layout()
plt.show()

print("📊 KEY ML INSIGHTS from pairplot:")
print("   ✅ Gentoo (green) completely SEPARABLE on flipper_length and body_mass")
print("   ✅ Adelie and Chinstrap overlap on most features except bill_depth_mm")
print("   → A simple Decision Tree on flipper_length alone would classify Gentoo perfectly")
print("   → Separating Adelie vs Chinstrap needs bill features — harder problem")

### 📋 Output Preview — pairplot with hue
```
3 colored clusters visible:
  🔵 Adelie:    medium bill, short flipper, medium mass
  🟠 Chinstrap: long bill, shallow depth, short flipper
  🟢 Gentoo:    long flipper, heaviest body — clearly separated

Best separating feature pairs:
  flipper × body_mass → Gentoo vs rest
  bill_length × bill_depth → Adelie vs Chinstrap
```
> 🎯 **Interview Q:** *"What is the difference between `diag_kind='hist'` and `diag_kind='kde'`?"*
> → `'hist'` shows binned bars — faster, shows exact counts. `'kde'` shows a smooth curve — better for comparing group distributions (less sensitive to bin width). With `hue=`, **kde is preferred** because overlapping KDE curves are easier to read than overlapping histograms.

In [ ]:
# 🟡 Intermediate | pairplot with corner=True — cleaner version
# Business Problem: Diamond pricing — which raw features predict price best?
# Using corner=True gives a cleaner look (no redundant upper panels).

diamond_sample = diamonds.sample(n=800, random_state=42)

g = sns.pairplot(
    data=diamond_sample[['carat', 'depth', 'table', 'price']],
    corner=True,                      # Only lower triangle — no redundancy
    diag_kind='kde',
    plot_kws={'alpha': 0.30, 's': 20, 'color': '#6A1B9A'},
    diag_kws={'color': '#6A1B9A', 'fill': True, 'alpha': 0.5},
    height=2.8
)

g.figure.suptitle(
    '💎 Diamond Analytics — Feature Relationships (corner=True)\n'
    'Showing only lower triangle — no redundant cells',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

# Correlation report
diamond_corr = diamond_sample[['carat','depth','table','price']].corr()['price'].drop('price')
print("📊 Correlation with diamond price:")
print(diamond_corr.sort_values(ascending=False).to_string())
print()
print("✅ carat is by far the strongest predictor of price.")
print("   depth and table have weak correlations — poor predictors alone.")

---

<a id='jointplot'></a>
## 5. `jointplot` — Bivariate Deep Dive

### 🧠 Intuition

`jointplot` gives you a **deep dive into ONE pair of variables** — simultaneously showing:
- The **joint distribution** (center panel)
- The **marginal distribution of X** (top panel)
- The **marginal distribution of Y** (right panel)

Where `pairplot` gives you breadth (all pairs, less detail), `jointplot` gives you depth (one pair, all details).

### 🔑 The `kind=` Parameter — 5 Options

| `kind=` | Center Plot | Best For |
|---------|-------------|---------|
| `'scatter'` | Scatter plot | Default — raw data |
| `'kde'` | 2D density contours | Dense data, showing distribution shape |
| `'hex'` | Hexagonal bins | Very large datasets (n > 10,000) |
| `'reg'` | Scatter + regression line | Confirming linear relationship |
| `'resid'` | Residual plot | Checking regression assumptions |

> ⚠️ **Important:** `jointplot` is **figure-level** — creates its own Figure. No `ax=` parameter.

In [ ]:
# 🟡 Intermediate | jointplot — all 5 kinds compared side by side
# We create 4 separate jointplots and manually position them.
# Business Problem: Analyzing bill vs tip relationship in full depth.

# ── kind='scatter' (default) ──
g1 = sns.jointplot(
    data=tips, x='total_bill', y='tip',
    kind='scatter',
    color='#1565C0',
    alpha=0.55,
    marginal_kws={'bins': 20, 'fill': True}
)
g1.figure.suptitle("kind='scatter' — Raw Data\n+ Marginal Histograms",
                    fontweight='bold', y=1.03, fontsize=11)
g1.set_axis_labels('Total Bill ($)', 'Tip ($)')
plt.tight_layout()
plt.show()

In [ ]:
# 🟡 Intermediate | jointplot kind='kde' — density contours
g2 = sns.jointplot(
    data=tips, x='total_bill', y='tip',
    kind='kde',
    fill=True,
    cmap='Blues',
    marginal_kws={'fill': True, 'color': '#1565C0', 'alpha': 0.6}
)
g2.figure.suptitle("kind='kde' — 2D Density Contours\n(Best for seeing where data concentrates)",
                    fontweight='bold', y=1.03, fontsize=11)
g2.set_axis_labels('Total Bill ($)', 'Tip ($)')
plt.tight_layout()
plt.show()

print("📊 KDE INSIGHT: The densest region (innermost contour) is around $15 bill, $2 tip.")
print("   This represents the 'typical' restaurant visit — not a special occasion.")

In [ ]:
# 🟡 Intermediate | jointplot kind='reg' — regression + marginals
# This combines regplot + KDE marginals in one unified view.
g3 = sns.jointplot(
    data=tips, x='total_bill', y='tip',
    kind='reg',
    scatter_kws={'alpha': 0.45, 's': 35, 'color': '#1565C0'},
    line_kws={'color': '#E53935', 'linewidth': 2.5},
    marginal_kws={'color': '#1565C0', 'fill': True, 'alpha': 0.5}
)
g3.figure.suptitle("kind='reg' — Scatter + Regression Line\n+ Marginal KDE",
                    fontweight='bold', y=1.03, fontsize=11)
g3.set_axis_labels('Total Bill ($)', 'Tip ($)')

# Annotate the Pearson r and p-value
corr_val = tips['total_bill'].corr(tips['tip'])
g3.figure.text(0.14, 0.72, f'r = {corr_val:.3f}', fontsize=11,
               color='#c62828', fontweight='bold',
               bbox=dict(boxstyle='round', fc='#FFEBEE', alpha=0.8))
plt.tight_layout()
plt.show()

In [ ]:
# 🔴 Advanced | jointplot kind='hex' — for large datasets
# Business Problem: Diamond carat vs price — 53,940 points.
# Scatter would be overplotted. Hexbin aggregates into hexagonal bins.

g4 = sns.jointplot(
    data=diamonds, x='carat', y='price',
    kind='hex',
    gridsize=35,               # Number of hexagons per axis
    cmap='YlOrRd',             # Light=few points | Dark=many points
    marginal_kws={'color': '#E53935', 'fill': True, 'alpha': 0.6}
)
g4.figure.suptitle("kind='hex' — Hexbin (Use for n > 10,000)\n💎 Diamond: carat vs price",
                    fontweight='bold', y=1.03, fontsize=11)
g4.set_axis_labels('Carat', 'Price ($)')
plt.tight_layout()
plt.show()

print("📊 HEXBIN INSIGHT: Darkest hexagons cluster at 0.2–0.5 carat, $500–$2,000 range.")
print("   This is the most common diamond purchase — small, affordable stones.")
print("   High-carat (>2.0) diamonds are very rare (light hexagons = few data points).")

### 📋 Output Preview — jointplot Summary
| kind | What you see | When to use |
|------|-------------|-------------|
| `'scatter'` | Individual dots + marginal histograms | n < 500, want raw data |
| `'kde'` | Smooth contour density + KDE margins | Any n, want density shape |
| `'reg'` | Dots + regression line + KDE margins | Confirming linearity |
| `'hex'` | Hexagonal bins (color=density) + histograms | n > 10,000 |
| `'resid'` | Residuals vs fitted + distributions | Regression diagnostics |

> 🎯 **Interview Q:** *"What does jointplot show that a regular scatter doesn't?"*
> → The **marginal distributions** on the top and right — you can see the shape of each variable individually while also seeing their joint relationship. It's 3 plots in 1.

---

<a id='clustermap'></a>
## 6. `clustermap` — Hierarchical Grouping

### 🧠 Intuition

`clustermap` is a **heatmap + hierarchical clustering** in one. It:
1. Computes similarity between rows and columns
2. Groups similar rows/columns together using a **dendrogram** (tree diagram)
3. Reorders the heatmap so that similar items appear next to each other

**Key use:** Finding natural groupings of features or observations that a regular heatmap would miss because of fixed ordering.

### 🔑 What is a Dendrogram?
The tree diagram on the top and left sides of the clustermap shows the **clustering hierarchy**:
- Items at the bottom of the tree are most similar
- Items at the top are most different
- The height of a branch = how different those groups are

In [ ]:
# 🟡 Intermediate | clustermap — feature grouping on correlation matrix
# Business Problem: Group automotive features by similarity.
# This reveals which features measure the "same thing" — identifying redundancy.

corr_mpg  = mpg.select_dtypes(include='number').corr()

g = sns.clustermap(
    corr_mpg,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    figsize=(10, 8),
    linewidths=0.5,
    annot_kws={'size': 9},
    cbar_pos=(0.02, 0.85, 0.03, 0.12),   # Position the colorbar
    dendrogram_ratio=0.15,               # Size of dendrogram relative to heatmap
)

g.figure.suptitle(
    '🚗 Automotive Features — Clustermap\n'
    'Dendrogram shows which features are most similar',
    fontsize=13, fontweight='bold', y=1.02
)
plt.show()

print("📊 CLUSTERMAP INSIGHT:")
print("   The dendrogram will group SIMILAR features together:")
print("   Group 1: weight, displacement, cylinders, horsepower (all 'power/size' features)")
print("   Group 2: mpg, efficiency (fuel economy features — NEGATIVE correlation with group 1)")
print("   Group 3: acceleration (loosely related — depends on both power and weight)")
print()
print("💡 This tells you: don't use weight + cylinders + displacement all together in one model.")
print("   They're measuring essentially the same construct — pick one representative.")

In [ ]:
# 🔴 Advanced | clustermap with standard_scale — normalize before clustering
# Business Problem: When features have very different scales,
# standardize before clustering to prevent scale from dominating.

# Select only numeric columns, sample for performance
peng_scaled = penguins.select_dtypes(include='number')

g2 = sns.clustermap(
    peng_scaled.corr(),
    method='ward',              # Ward linkage — minimizes within-cluster variance
    metric='euclidean',
    standard_scale=1,           # Standardize columns (z-score) before clustering
    annot=True,
    fmt='.2f',
    cmap='vlag',                # Diverging colormap designed for heatmaps
    center=0,
    figsize=(8, 7),
    linewidths=0.8,
    dendrogram_ratio=0.18,
    annot_kws={'size': 10},
)

g2.figure.suptitle(
    '🐧 Penguin Feature Similarity — Normalized Clustermap\n'
    '(method=ward | standard_scale=1)',
    fontsize=12, fontweight='bold', y=1.02
)
plt.show()

print("📊 WHEN TO USE standard_scale=1:")
print("   When features have different scales (e.g., bill_length in mm vs body_mass in grams).")
print("   Without scaling, the feature with the largest scale dominates the clustering.")
print("   → Always normalize when units differ across features.")

---

<a id='feat-corr'></a>
## 7. Feature Correlation Ranking — Core ML Skill

### 🧠 Why This Matters

In Machine Learning, **feature selection** is one of the first steps after EDA. One fast method:
1. Compute correlation of each feature **with the target variable**
2. Rank by absolute value (positive and negative correlations both matter)
3. Visualize as a horizontal bar chart
4. Use the top features to build your first model

> ⚠️ **Limitation:** This only catches **linear** correlations. Non-linear relationships are invisible to Pearson r. But for a first pass, it's fast and effective.

> 🎯 **Interview Gold:** Interviewers love when freshers say: *"I first checked feature-target correlations using df.corr(), then validated non-linear patterns with pairplot and regplot(lowess=True)."*

In [ ]:
# 🟡 Intermediate | Feature correlation ranking — ML feature selection
# Business Problem: Building a tip prediction model.
# Which features should we include? Start with correlation ranking.

# ── All-feature correlation with target ──
tips_numeric_full = tips[['total_bill', 'tip', 'size', 'tip_pct', 'table_size']]
corr_with_tip = (
    tips_numeric_full.corr()['tip']
    .drop('tip')
    .abs()
    .sort_values(ascending=True)   # Ascending for horizontal bar (largest at top)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Left: Horizontal bar chart — feature importance proxy ──
colors_feat = ['#E53935' if v > 0.5 else '#FB8C00' if v > 0.3 else '#FDD835'
               for v in corr_with_tip.values]
bars = axes[0].barh(
    corr_with_tip.index,
    corr_with_tip.values,
    color=colors_feat,
    edgecolor='white'
)
axes[0].bar_label(bars, fmt='%.3f', fontsize=10, fontweight='bold', padding=4)
axes[0].set_title(
    '🍽️ Feature Correlation with Tip Amount\n'
    '(Absolute Pearson r | Red > 0.5 | Orange > 0.3)',
    fontweight='bold', fontsize=11
)
axes[0].set_xlabel('|Pearson r| with tip')
axes[0].axvline(x=0.5, color='#E53935', linestyle='--', alpha=0.6, label='Strong (0.5)')
axes[0].axvline(x=0.3, color='#FB8C00', linestyle='--', alpha=0.6, label='Moderate (0.3)')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 0.85)

# ── Right: Same for MPG dataset — automotive feature importance ──
mpg_numeric_cols = ['cylinders', 'displacement', 'horsepower', 'weight',
                    'acceleration', 'model_year', 'mpg']
corr_with_mpg = (
    mpg[mpg_numeric_cols].corr()['mpg']
    .drop('mpg')
    .sort_values(key=abs, ascending=True)
)

colors_mpg = ['#E53935' if abs(v) > 0.7 else '#FB8C00' if abs(v) > 0.4 else '#FDD835'
              for v in corr_with_mpg.values]
bars2 = axes[1].barh(
    corr_with_mpg.index,
    corr_with_mpg.abs().values,
    color=colors_mpg,
    edgecolor='white'
)
axes[1].bar_label(bars2, fmt='%.3f', fontsize=10, fontweight='bold', padding=4)
axes[1].set_title(
    '🚗 Feature Correlation with MPG\n'
    '(Top features for fuel efficiency model)',
    fontweight='bold', fontsize=11
)
axes[1].set_xlabel('|Pearson r| with mpg')
axes[1].axvline(x=0.7, color='#E53935', linestyle='--', alpha=0.6, label='Strong (0.7)')
axes[1].axvline(x=0.4, color='#FB8C00', linestyle='--', alpha=0.6, label='Moderate (0.4)')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 0.95)

fig.suptitle('🎯 ML Feature Selection — Correlation-Based Ranking',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("📊 TIP PREDICTION MODEL — Feature Recommendation:")
print(f"   Use 'total_bill' first (r={tips_numeric_full.corr().loc['total_bill','tip']:.3f}) — strongest predictor")
print(f"   Add 'size' second  (r={tips_numeric_full.corr().loc['size','tip']:.3f}) — moderate predictor")
print(f"   Avoid 'tip_pct'   (derived from tip itself — data leakage!)")
print()
print("📊 MPG PREDICTION MODEL — Feature Recommendation:")
best_mpg_feat = corr_with_mpg.abs().idxmax()
print(f"   Best single predictor: '{best_mpg_feat}' (r={corr_with_mpg.abs().max():.3f})")

### 📋 Output Preview — Feature Ranking
```
📊 TIP PREDICTION MODEL:
   total_bill: |r| = 0.676  ← Start here
   size:       |r| = 0.489  ← Add second
   table_size: |r| = 0.489  ← Same as size (engineered feature)
   tip_pct:    |r| = ~0.0   ← Derived from tip — data leakage!

📊 MPG PREDICTION MODEL:
   Best single predictor: 'weight' (|r| = 0.832)
```
> 🎯 **Interview Q:** *"What is data leakage and how does it appear in feature correlation?"*
> → Data leakage is when a feature contains information from the **target variable** itself. `tip_pct = tip / total_bill` is derived FROM tip, so using it to predict tip is cheating — your model would appear perfect in training but fail on new data. **Always check how engineered features are computed.**

---

<a id='dashboard'></a>
## 8. Business Dashboard — Multivariate EDA in Practice

In [ ]:
# 🔴 Advanced | Full multivariate EDA dashboard
# Business Goal: Complete multivariate analysis of the Titanic dataset.
# Answer 4 key questions with 4 different multivariate plots.

titanic_num = titanic[['survived', 'age', 'fare', 'pclass', 'sibsp', 'parch']].dropna()
corr_titanic = titanic_num.corr()
mask_t       = np.triu(np.ones_like(corr_titanic, dtype=bool))

fig, axes = plt.subplots(2, 2, figsize=(16, 13))

# ── Panel 1: Masked Correlation Heatmap ──
sns.heatmap(
    corr_titanic,
    mask=mask_t,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    square=True, linewidths=0.8,
    cbar_kws={'shrink': 0.7},
    ax=axes[0, 0]
)
axes[0, 0].set_title(
    '① Correlation Matrix (masked)\n→ pclass strongly negatively correlated with survival',
    fontweight='bold', fontsize=10
)
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=30, ha='right', fontsize=9)

# ── Panel 2: Missing Value Heatmap ──
sns.heatmap(
    titanic[['age', 'fare', 'embarked', 'deck', 'survived']].isnull(),
    yticklabels=False,
    cmap='gray_r',
    cbar=False,
    ax=axes[0, 1]
)
axes[0, 1].set_title(
    '② Missing Value Map\n→ deck mostly missing, age partially missing',
    fontweight='bold', fontsize=10
)

# ── Panel 3: Feature Correlation with Survival ──
surv_corr = (
    corr_titanic['survived']
    .drop('survived')
    .sort_values(key=abs, ascending=True)
)
colors_surv = ['#E53935' if abs(v) > 0.3 else '#FB8C00' if abs(v) > 0.15 else '#FDD835'
               for v in surv_corr.values]
bars3 = axes[1, 0].barh(surv_corr.index, surv_corr.abs().values,
                         color=colors_surv, edgecolor='white')
axes[1, 0].bar_label(bars3, fmt='%.3f', fontsize=10, fontweight='bold', padding=4)
axes[1, 0].set_title(
    '③ Feature Correlation with Survival\n→ pclass is the strongest predictor',
    fontweight='bold', fontsize=10
)
axes[1, 0].set_xlabel('|Pearson r| with survived')

# ── Panel 4: Survival rate by class — barplot ──
survival_rate = titanic.groupby('pclass')['survived'].mean().reset_index()
survival_rate.columns = ['pclass', 'survival_rate']
survival_rate['pclass'] = survival_rate['pclass'].map({1: '1st Class', 2: '2nd Class', 3: '3rd Class'})

b4 = axes[1, 1].bar(
    survival_rate['pclass'],
    survival_rate['survival_rate'] * 100,
    color=['#2E7D32', '#FB8C00', '#E53935'],
    edgecolor='white', linewidth=1.5, width=0.5
)
pct_1st = f"{survival_rate['survival_rate'].iloc[0]*100:.1f}%"
pct_2nd = f"{survival_rate['survival_rate'].iloc[1]*100:.1f}%"
pct_3rd = f"{survival_rate['survival_rate'].iloc[2]*100:.1f}%"
axes[1, 1].bar_label(b4, labels=[pct_1st, pct_2nd, pct_3rd],
                      fontsize=12, fontweight='bold', padding=4)
axes[1, 1].set_title(
    '④ Survival Rate by Passenger Class\n→ 1st class survival rate 3× higher than 3rd',
    fontweight='bold', fontsize=10
)
axes[1, 1].set_ylabel('Survival Rate (%)')
axes[1, 1].set_ylim(0, 80)

fig.suptitle('🚢 Titanic Multivariate EDA Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/titanic_multivariate_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Dashboard saved: outputs/titanic_multivariate_dashboard.png")
print()
print("📊 KEY FINDINGS:")
print(f"   1st class survival: {survival_rate[survival_rate['pclass']=='1st Class']['survival_rate'].values[0]*100:.1f}%")
print(f"   3rd class survival: {survival_rate[survival_rate['pclass']=='3rd Class']['survival_rate'].values[0]*100:.1f}%")
print(f"   pclass ↔ survived: r = {corr_titanic.loc['pclass','survived']:.3f} (most predictive feature)")

---

<a id='best-practices'></a>
## 9. Best Practices

| Plot | Best Practice |
|------|--------------|
| **heatmap** | Always mask upper triangle with `np.triu` |
| **heatmap** | Always use `center=0` with `coolwarm` — no center = misleading colors |
| **heatmap** | Always `select_dtypes(include='number')` before calling `.corr()` |
| **heatmap** | Use `fmt='.2f'` not `fmt='d'` for correlation (it's a float) |
| **pairplot** | Limit to 6–8 variables max — beyond that, `vars=` to select key ones |
| **pairplot** | Use `diag_kind='kde'` with `hue=` — histograms overlap poorly |
| **pairplot** | Use `corner=True` for cleaner look in presentations |
| **jointplot** | Use `kind='hex'` for n > 10,000 — scatter is unreadable |
| **jointplot** | Use `kind='reg'` to combine scatter + trend in one figure |
| **clustermap** | Always use `standard_scale=1` when features have different units |
| **Feature ranking** | Sort by `abs()` — both positive and negative correlations matter |
| **Feature ranking** | Always check for data leakage before adding engineered features |

---

<a id='common-mistakes'></a>
## 10. Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| Calling `.corr()` without `select_dtypes` | Error on categorical columns | `df.select_dtypes(include='number').corr()` |
| Not using `mask=` in heatmap | Redundant upper triangle confuses readers | Always `mask = np.triu(np.ones_like(corr, dtype=bool))` |
| Not setting `center=0` in heatmap | Color scale off-center — misleading perception | Always `center=0` with `coolwarm` |
| Using `fmt='d'` with correlation values | Correlation r values are floats — 'd' rounds to 0 | Use `fmt='.2f'` |
| Running pairplot on all columns | Includes categoricals → error or meaningless panels | Use `vars=['col1','col2',...]` or `select_dtypes` |
| Using `jointplot` inside `plt.subplots()` | Figure-level: can't pass `ax=` | Use standalone or access `.ax_joint` for axes-level tricks |
| Treating high r as causation | Correlation ≠ causation | Always note: *"correlated, not necessarily causal"* |
| Forgetting that clustermap reorders axes | Comparing positions with original matrix | Read the dendrogram, not the position |

---

<a id='interview-notes'></a>
## 11. 🎤 Interview Notes

---

**Q: How do you mask the upper triangle of a correlation heatmap?**
> *"I use `np.triu` to create a boolean mask where `True` means hide that cell. The key line is:*
> `mask = np.triu(np.ones_like(corr, dtype=bool))`
> *Then I pass it to `sns.heatmap(corr, mask=mask, ...)`. The upper triangle is a mirror of the lower triangle, so showing both is redundant."*

---

**Q: What is multicollinearity and how do you detect it in Seaborn?**
> *"Multicollinearity is when two or more predictor variables are highly correlated with each other (not just the target). It makes linear regression coefficients unstable. I detect it with a correlation heatmap — any correlation above 0.8 between two features is a warning sign. I then verify using Variance Inflation Factor (VIF) for confirmation."*

---

**Q: What is the difference between pairplot and clustermap?**
> *"pairplot shows scatter and distribution plots for all variable pairs — it's great for understanding relationships and spotting class separation. clustermap applies hierarchical clustering to group similar rows or columns together and reorders the heatmap accordingly — it's better for finding natural feature groupings and identifying redundant features."*

---

**Q: What does jointplot show that a pairplot doesn't?**
> *"jointplot provides a deeper look at ONE variable pair — showing the joint distribution (center) plus the marginal distribution of each variable (top and right). It also offers multiple `kind=` options including KDE contours, hexbin density, and regression lines. pairplot gives breadth (all pairs), jointplot gives depth (one pair, full detail)."*

---

**Q: How do you use correlation for feature selection?**
> *"I compute `df.corr()['target_column'].drop('target_column').abs().sort_values(ascending=False)`. This ranks all features by their absolute Pearson correlation with the target. I then visualize this as a horizontal bar chart. Features with |r| > 0.5 are strong candidates, 0.3–0.5 are moderate. I always also check for data leakage — engineered features derived from the target will show artificially high correlation."*

---

<a id='summary'></a>
## 12. Summary & Key Takeaways

| Plot | One-Line Summary |
|------|-----------------|
| `heatmap` | Color-coded matrix of correlations — always mask upper triangle |
| Missing value heatmap | `df.isnull()` → visually audit what's missing before imputation |
| `pairplot` | All-vs-all scatter + distributions — fastest full EDA overview |
| `jointplot` | Deep single-pair analysis with 5 visualization kinds |
| `clustermap` | Heatmap + hierarchical clustering — reveals natural feature groups |

### 🏆 The 4 Lines Every Fresher Must Know Cold

```python
# 1. Masked correlation heatmap — THE pattern
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)

# 2. Missing value audit
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='gray_r')

# 3. Quick pairplot with hue
sns.pairplot(df, hue='species', diag_kind='kde', palette='colorblind')

# 4. Feature-target correlation ranking
df.corr()['target'].drop('target').abs().sort_values(ascending=False)
```

---

<a id='practice'></a>
## 13. 📝 Practice Questions

---

### 🟢 Easy (1–5)

**Q1.** Load the `mpg` dataset and create a masked correlation heatmap using `coolwarm`. Use `annot=True, fmt='.2f', square=True`. Which two features have the highest positive correlation?

**Q2.** Load the `titanic` dataset (with NaN). Create a missing value heatmap using `df.isnull()` with `cmap='gray_r'`. Which column has the most missing values?

**Q3.** Create a `pairplot` of the `penguins` dataset using only `['bill_length_mm', 'flipper_length_mm', 'body_mass_g']` with `hue='species'` and `diag_kind='kde'`. Which species is most clearly separated from the others?

**Q4.** Create a `jointplot` for the `diamonds` dataset with `x='carat'`, `y='price'`, and `kind='hex'`. Use a sample of 5,000 rows. What does the hexagon color represent?

**Q5.** Write the 4 lines to compute and display the features most correlated with `survived` in the `titanic` dataset. Explain what `abs()` does in this context.

---

### 🟡 Medium (6–11)

**Q6.** Create a 1×2 figure showing:
- Left: Full (unmasked) correlation heatmap of `mpg` numeric features
- Right: Masked lower-triangle version
Write a comment explaining why the right version is the professional standard.

**Q7.** Using `tips`, build a feature correlation ranking bar chart with `tip` as the target. Color bars: red if |r| > 0.5, orange if 0.3–0.5, yellow otherwise. Add value labels on each bar.

**Q8.** Create a `pairplot` of the `penguins` dataset with `corner=True` and `diag_kind='kde'`. Then add a title to the figure. Explain the difference between `corner=True` and a regular pairplot.

**Q9.** Create 3 `jointplot` charts for `total_bill` vs `tip` from `tips` with `kind='scatter'`, `kind='kde'`, and `kind='reg'`. For `kind='reg'`, annotate the Pearson r value on the joint axes.

**Q10.** Using `mpg`, create a `clustermap` of the correlation matrix with `method='ward'` and `standard_scale=1`. Which features cluster together? What does this tell you about feature redundancy for an ML model?

**Q11.** For the `titanic` dataset, compute missing percentage per column (using `df.isnull().mean() * 100`), then create a horizontal bar chart showing only columns with missing > 0%. Color bars red if > 30%, orange if 5–30%, yellow otherwise.

---

### 🔴 Hard (12–15)

**Q12. 🏗️ Complete EDA Report — Diamonds Dataset.**
Using `diamonds`, build a 2×2 multivariate dashboard:
- Panel 1: Masked correlation heatmap of all numeric features
- Panel 2: Feature correlation ranking with `price` as target
- Panel 3: `pairplot` (just the carat vs price panel extracted from jointplot)
- Panel 4: jointplot `kind='hex'` for carat vs price
Write 3 business insights from the combined analysis.

**Q13. 🔍 Multicollinearity Investigation.**
Using `mpg`, identify all pairs of features with |r| > 0.8.
For each such pair:
1. Print the pair and correlation value
2. Create a `jointplot` showing their relationship
3. Explain which feature you would DROP from a linear regression and why.
Do this without using any loops — use Pandas boolean indexing on the correlation matrix.

**Q14. 🎨 Group Separability Analysis.**
Using `penguins`, answer: *"Which single feature best separates the three species?"*
Method:
1. Compute mean of each feature grouped by species
2. Compute variance of each feature overall
3. Compute **between-group variance / within-group variance** (F-statistic proxy) using Pandas `.groupby().var()` — no sklearn
4. Rank features by this ratio
5. Visualize the top-2 features in a `jointplot` with `hue='species'`

**Q15. 🏆 Full Titanic EDA Story.**
Using the `titanic` dataset, create a complete multivariate narrative:
1. Missing value audit (heatmap)
2. Feature correlation with survival (horizontal bar)
3. Masked correlation heatmap of all numeric features
4. `pairplot` of age, fare, pclass, survived with `hue='survived'`
5. Write a final markdown cell with 5 data-driven insights that would guide feature selection for a survival prediction model.

---

### 🔥 Predict the Output Challenge

Study this code carefully:

```python
corr = tips.select_dtypes(include='number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True,
            vmin=-1, vmax=1)
```

❓ Questions (answer before running):
1. How many cells will be shown vs hidden?
2. What color will a cell with value `+0.68` appear?
3. What color will a cell with value `-0.14` appear?
4. What does `center=0` do to the colormap?
5. What would happen if you used `fmt='d'` instead of `fmt='.2f'`?

---

<div style="background: linear-gradient(135deg, #0d1b2a 0%, #1b2f4b 100%); padding: 30px; border-radius: 12px; text-align: center; margin-top: 40px;">
  <h2 style="color: #7ec8e3; font-size: 1.8em;">🎉 Notebook 4 Complete!</h2>
  <p style="color: #e0f2fe; font-size: 1.1em;">You can now do full multivariate EDA — correlation, missing values, pairplots, and feature selection.</p>
  <hr style="border: 1px solid #7ec8e3; margin: 15px auto; width: 60%;">
  <p style="color: #ccc;">📚 Next: <strong style='color:#e0f2fe'>Notebook 5</strong> — Advanced Visualization</p>
  <p style="color: #aaa; font-size: 0.9em;">FacetGrid · PairGrid · Custom themes · Annotations · Reusable dashboard functions</p>
</div>